# SQL Queries (BigQuery) for Audi A3 Germany

This notebook runs example SQL queries against the BigQuery dataset and produces simple plots for class discussion.

These are example queries for classroom discussion. They do not write tables, but they must remain identical because later notebooks assume the same schema and naming.

### Detailed explanation
This notebook shows how to query the BigQuery dataset with SQL. Each section demonstrates a common pattern: joins, filters, aggregations, and simple visualizations. These queries are read-only and are used to build intuition about the data.

### Functional overview
Inputs: BigQuery tables in the autoscout dataset.
Process: run read-only SQL queries with joins, filters, and aggregations.
Outputs: displayed tables and plots that help explain the dataset.
Reason: these examples teach students how to explore data stored in BigQuery.

**Objective:** Demonstrate read-only SQL queries for exploring the dataset.
**Inputs:** BigQuery tables in the autoscout dataset.
**Process:** Run joins, filters, aggregations, and simple visualizations.
**Outputs:** Displayed tables and charts for classroom discussion.
**Why:** SQL exploration teaches how the modeled tables are connected.


## 1. Imports & config

We keep configuration together so students can change project or dataset in one place.

### Configuration
We set the project and dataset once, then reuse them in every query. This keeps queries consistent and easy to update for different environments.

### Configuration purpose
We define the project and dataset once, which reduces errors and keeps SQL consistent.

**Objective:** Set global BigQuery configuration and client.
**Inputs:** Project and dataset IDs.
**Process:** Initialize a BigQuery client.
**Outputs:** A client object used by all queries.
**Why:** Central config avoids repeating IDs in every cell.


In [ ]:
# Objective: Import libraries and define BigQuery configuration.  # Objective
# Input: bigquery client for SQL execution, pandas for tables, matplotlib for plots.  # Input
# Input: Shared project_config.yaml for project, dataset, and scope defaults.  # Input
# Process: Import libraries, load config, set IDs, and create a BigQuery client.  # Process
# Output: client connected to BigQuery and reusable configuration variables.  # Output

from pathlib import Path
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


REPO_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(REPO_ROOT / 'config' / 'project_config.yaml')

project_id = str(PROJECT_CONFIG.get('gcp_project_id', 'albertheadofdata101')).strip()
dataset_id = str(PROJECT_CONFIG.get('bq_dataset', 'autoscout_audi_a3_germany')).strip()
make = str(PROJECT_CONFIG.get('make', 'audi')).strip().lower()
model = str(PROJECT_CONFIG.get('model', 'a3')).strip().lower()
country_name = str(PROJECT_CONFIG.get('country', 'germany')).strip().lower()

client = bigquery.Client(project=project_id)


## 2. Helper to run SQL

The helper reduces repeated code and keeps each query cell short.

### Helper function
The run_query helper keeps every query cell short by hiding the job execution and DataFrame conversion.

### Helper purpose
The helper function removes repeated boilerplate and makes each SQL cell easy to read.

**Objective:** Create a helper to run SQL and return DataFrames.
**Inputs:** SQL query strings.
**Process:** Execute query and download results.
**Outputs:** pandas DataFrames for display and plotting.
**Why:** Reduces boilerplate and keeps query cells short.


In [ ]:
# Objective: Define a helper to run SQL and return a DataFrame.  # Objective
# Input: SQL string passed to the function.  # Input
# Process: Submit query to BigQuery and download results as pandas.  # Process
# Output: DataFrame containing the query results.  # Output

def run_query(sql):  # Define a helper that takes a SQL string.
    query_job = client.query(sql)  # Submit the SQL query to BigQuery.
    return query_job.to_dataframe()  # Download results as a pandas DataFrame.


## 3. SQL assets and dataset preview


In [ ]:
# Objective: Validate SQL asset order and preview the regression dataset view.  # Objective
# Input: SQL files in sql/ and configured project/dataset scope.  # Input
# Process: List required SQL files, check existence, then query vw_regression_dataset.  # Process
# Output: Asset validation printout and df_q1 preview table.  # Output

sql_dir = REPO_ROOT / 'sql'
ordered_sql_files = [
    '00_create_dataset.sql',
    '01_create_staging.sql',
    '02_build_dimensions.sql',
    '03_build_fact.sql',
    '04_vw_regression_dataset.sql',
    '05_vw_classification_dataset.sql',
    '06_vw_bi_dashboard.sql',
]

for file_name in ordered_sql_files:
    file_path = sql_dir / file_name
    status = 'OK' if file_path.exists() else 'MISSING'
    print(f'{status}: {file_name}')

sql_query_1 = f'''
SELECT
  listing_id,
  make,
  model,
  listing_country,
  actual_price_eur,
  mileage_km,
  age_years,
  power_hp
FROM `{project_id}.{dataset_id}.vw_regression_dataset`
WHERE LOWER(make) = '{make}'
  AND LOWER(model) = '{model}'
  AND LOWER(listing_country) = '{country_name}'
LIMIT 20
'''

df_q1 = run_query(sql_query_1)
df_q1


## 4. Filtered query (country + fuel)

This query shows how to apply filters using readable labels from dimensions.

### Filtering example
We filter by country and fuel type to show how WHERE clauses work with dimension attributes.

### Query purpose
This query shows how to filter by country and fuel type using WHERE clauses.

**Objective:** Show filtering using dimension attributes.
**Inputs:** Filter values for country and fuel type.
**Process:** Apply WHERE clauses in SQL.
**Outputs:** Filtered listing sample.
**Why:** Demonstrates how to subset data in BigQuery.


In [ ]:
# Objective: Run a filtered query by country and fuel type.  # Objective
# Input: filter_country and filter_fuel control the WHERE clause.  # Input
# Process: Build SQL with filters, execute, and display results.  # Process
# Output: df_q2 containing filtered listing rows.  # Output

filter_country = country_name
filter_fuel = 'diesel'

sql_query_2 = f'''
SELECT
  dm.make AS make,
  dm.model AS model,
  df.fuel_type AS fuel_type,
  dc.listing_country AS listing_country,
  dpl.price_label AS price_label,
  fl.price_eur AS price_eur,
  fl.power_hp AS power_hp
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_fuel` AS df
  ON fl.fuel_id = df.fuel_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
JOIN `{project_id}.{dataset_id}.dim_price_label` AS dpl
  ON fl.price_label_id = dpl.price_label_id
WHERE dc.listing_country = '{filter_country}'
  AND df.fuel_type = '{filter_fuel}'
LIMIT 20
'''

df_q2 = run_query(sql_query_2)
df_q2


## 5. Top prices for a single model

This query focuses on a single model to illustrate ordering and limits.

### Ordering example
We order by price to find the most expensive listings for one model.

### Query purpose
This query finds the most expensive listings for one model by ordering on price.

**Objective:** Retrieve top-priced listings for a model.
**Inputs:** Target model name.
**Process:** Order by price descending and limit results.
**Outputs:** Top 10 expensive listings.
**Why:** Shows ordering and ranking in SQL.


In [ ]:
# Objective: List the top 10 prices for the configured model scope.  # Objective
# Input: model from project_config.yaml determines which model is analyzed.  # Input
# Process: Build SQL, order by price descending, limit to 10.  # Process
# Output: df_q3 with the most expensive listings for the model.  # Output

target_model = model

sql_query_3 = f'''
SELECT
  dm.make AS make,
  dm.model AS model,
  dpl.price_label AS price_label,
  fl.price_eur AS price_eur,
  fl.mileage_km AS mileage_km,
  fl.power_hp AS power_hp,
  df.fuel_type AS fuel_type,
  dc.listing_country AS listing_country
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_fuel` AS df
  ON fl.fuel_id = df.fuel_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
JOIN `{project_id}.{dataset_id}.dim_price_label` AS dpl
  ON fl.price_label_id = dpl.price_label_id
WHERE dm.model = '{target_model}'
ORDER BY fl.price_eur DESC
LIMIT 10
'''

df_q3 = run_query(sql_query_3)
df_q3


## 6. Listing counts by model and price label

Counting by label is a simple way to check distribution across pricing categories.

### Aggregation example
We count listings by model and price label to see distribution across categories.

### Query purpose
This query counts listings by model and price label to show distribution.

**Objective:** Count listings by model and price label.
**Inputs:** fact_listings and price label dimension.
**Process:** Group and count.
**Outputs:** Listing counts per group.
**Why:** Aggregations summarize distribution patterns.


In [ ]:
# Objective: Count listings by model and price label.  # Objective
# Input: fact and dimension tables in BigQuery.  # Input
# Process: Group by model and price label, count rows, sort by count.  # Process
# Output: df_q4 with counts per model and price label.  # Output

sql_query_4 = f'''  # Start SQL query string for counts.
SELECT
  dm.model AS model,
  dpl.price_label AS price_label,
  COUNT(*) AS total_listings
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_price_label` AS dpl
  ON fl.price_label_id = dpl.price_label_id
GROUP BY dm.model, dpl.price_label
ORDER BY total_listings DESC
'''  # End SQL query string.

df_q4 = run_query(sql_query_4)  # Execute aggregation query.
df_q4.head(10)  # Show the top 10 rows.


In [ ]:
# Objective: Visualize the top 10 models by listing count.  # Objective
# Input: df_q4 from the previous aggregation query.  # Input
# Process: Select top 10 rows and plot a bar chart.  # Process
# Output: A bar chart showing the most common models.  # Output

# Plot the top 10 models by number of listings  # Use the aggregation output.
top10_models = df_q4.head(10)  # Take the top 10 models by count.

plt.figure(figsize=(10, 4))  # Create a figure with a fixed size.
plt.bar(top10_models['model'], top10_models['total_listings'])  # Plot counts by model.
plt.title('Top 10 Models by Number of Listings')  # Set the chart title.
plt.xlabel('Model')  # Label the x-axis.
plt.ylabel('Number of Listings')  # Label the y-axis.
plt.xticks(rotation=45)  # Rotate labels for readability.
plt.tight_layout()  # Adjust layout to prevent overlap.
plt.show()  # Display the plot.


## 7. Listing counts by model, country, and price label

Adding country shows how distributions vary across markets.

### Grouping by more dimensions
Adding country shows how counts change across markets.

### Query purpose
This query adds country to the grouping so we can compare markets.

**Objective:** Count listings by model, country, and label.
**Inputs:** fact_listings and country/label dimensions.
**Process:** Group by multiple dimensions and count.
**Outputs:** Multi-dimensional counts table.
**Why:** Highlights differences across markets.


In [ ]:
# Objective: Count listings by model, country, and price label.  # Objective
# Input: fact and dimension tables in BigQuery.  # Input
# Process: Group by model, country, price label; count and order.  # Process
# Output: df_q5 with counts by model/country/label.  # Output

sql_query_5 = f'''  # Start SQL query string for multi-dimension counts.
SELECT
  dm.model AS model,
  dc.listing_country AS listing_country,
  dpl.price_label AS price_label,
  COUNT(*) AS total_listings
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
JOIN `{project_id}.{dataset_id}.dim_price_label` AS dpl
  ON fl.price_label_id = dpl.price_label_id
GROUP BY dm.model, dc.listing_country, dpl.price_label
ORDER BY dm.model ASC, total_listings DESC
'''  # End SQL query string.

df_q5 = run_query(sql_query_5)  # Execute the grouped query.
df_q5.head(20)  # Display a sample of rows.


## 8. Average price and power by model and country

Aggregate metrics are useful for comparisons and quick insights.

### Average metrics
Averages give a compact view of price and power differences across countries.

### Query purpose
This query computes average price and power by model and country for comparisons.

**Objective:** Compute average price and power by model and country.
**Inputs:** fact_listings with numeric fields.
**Process:** Average and round results.
**Outputs:** Table of mean price and power.
**Why:** Aggregates provide quick comparisons for class.


In [ ]:
# Objective: Compute average price and power by model and country.  # Objective
# Input: fact and dimension tables in BigQuery.  # Input
# Process: Group by model/country, compute averages, order by average price.  # Process
# Output: df_q6 with average price and power per group.  # Output

sql_query_6 = f'''  # Start SQL query string for averages.
SELECT
  dm.model AS model,
  dc.listing_country AS listing_country,
  ROUND(AVG(fl.price_eur), 2) AS avg_price_eur,
  ROUND(AVG(fl.power_hp), 2) AS avg_power_hp
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
GROUP BY dm.model, dc.listing_country
ORDER BY avg_price_eur DESC
'''  # End SQL query string.

df_q6 = run_query(sql_query_6)  # Execute average query.
df_q6.head(20)  # Display the top rows.


In [ ]:
# Objective: Plot average price by country for a single model.  # Objective
# Input: df_q6 containing averages by model and country.  # Input
# Process: Filter for one model, build a bar chart of average prices.  # Process
# Output: A bar chart comparing average prices across countries.  # Output

# Visualize the average price for a single model across countries  # Use aggregated results.
model_for_plot = 'a3'  # Choose the model to visualize.

df_model = df_q6[df_q6['model'] == model_for_plot]  # Filter to one model.

plt.figure(figsize=(8, 4))  # Create a figure with a fixed size.
plt.bar(df_model['listing_country'], df_model['avg_price_eur'])  # Plot average price by country.
plt.title(f'Average Price for {model_for_plot.upper()} by Country')  # Title with model name.
plt.xlabel('Country')  # Label the x-axis.
plt.ylabel('Average Price (EUR)')  # Label the y-axis.
plt.xticks(rotation=45)  # Rotate labels for readability.
plt.tight_layout()  # Adjust layout to avoid overlap.
plt.show()  # Display the plot.
